In [ ]:
# Keep huggingface_hub below 2.0: Kaggle's transformers needs <2.0.
# After this cell runs the first time, use Run > Restart & Run All (or just Save & Run All).
!pip install -q "huggingface_hub>=1.3,<2.0"


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")

In [ ]:
from huggingface_hub import login
login(hf_token)

In [ ]:
# ==========================================================
# CACHE CONFIG  -  reuse the fine-tuned BERT instead of retraining
# ==========================================================
# The ~15 min cost is the dataset-augmentation cell + the BERT training
# loop below. After training once, the model is pushed to the Hugging
# Face Hub; on every later run it is pulled back and both of those cells
# skip themselves.
#
#   FORCE_RETRAIN = False   -> ignore all caches, rebuild dataset + retrain
#   FORCE_RETRAIN = False  -> reuse the cached model whenever one exists
# ==========================================================
# RUN SELECTOR  -  change ONE line per run, then Save & Run All
#   "main"           -> soft targets ON,  contrastive ON,  upweight 6.0
#   "soft_off"       -> soft targets OFF, contrastive ON,  upweight 6.0
#   "no_contrastive" -> soft targets ON,  contrastive OFF, upweight 6.0
#   "soft_uw1"       -> soft targets ON,  contrastive ON,  upweight 1.0
#
# soft_uw1 exists to SEPARATE the two things the soft-label arm changes at
# once: the target representation (distribution vs majority vote) and the
# disagreement upweighting. main vs soft_uw1 isolates the weighting;
# soft_uw1 vs soft_off isolates the targets. Without it the soft-label
# ablation is a two-factor intervention reported as one.
# ==========================================================
RUN_LABEL  = "main"      # <<<<<< CHANGE THIS PER RUN <<<<<<
TRAIN_SEED = 0           # <<<<<< AND THIS: 0, 1, 2 <<<<<<
#
# TRAIN_SEED controls ONLY training stochasticity (weight init, shuffling,
# dropout). The data split is governed by SPLIT_SEED below and is held FIXED at
# 42 for every arm and every seed, so all 12 runs are evaluated on byte-identical
# dev and test partitions. One seed per arm cannot show whether a
# difference survives training randomness; three can.

_RUN_CFG = {
    "main":           dict(soft=True,  contrastive=True,  upweight=6.0),
    "soft_off":       dict(soft=False, contrastive=True,  upweight=6.0),
    "no_contrastive": dict(soft=True,  contrastive=False, upweight=6.0),
    "soft_uw1":       dict(soft=True,  contrastive=True,  upweight=1.0),
}
if RUN_LABEL not in _RUN_CFG:
    raise ValueError(f"RUN_LABEL must be one of {list(_RUN_CFG)}, got {RUN_LABEL!r}")

SOFT_LABEL_TRAINING = _RUN_CFG[RUN_LABEL]["soft"]   # HateXplain per-annotator label distributions
                             # (class-weighted soft cross-entropy), TESTING whether the
                             # probability bars end up modelling annotator disagreement.
                             # So far they do not (entropy_corr ~ 0.06 - see SOFT-LABEL
                             # ANALYSIS cell): report this as a negative result, not a
                             # "spectrum" claim, unless a later run changes that.
                             # False -> original hard-label CE.
SOFT_LABEL_ALPHA    = 1.0    # 1.0 = pure soft CE; < 1 blends in hard CE on the majority label
SOFT_DISAGREEMENT_UPWEIGHT = _RUN_CFG[RUN_LABEL]["upweight"]
                                   # weight on rows where annotators actually disagreed.
                                   # Driven by the RUN SELECTOR: 6.0 for main/soft_off/
                                   # no_contrastive, 1.0 for soft_uw1. Comparing main
                                   # against soft_uw1 isolates this factor from the
                                   # target representation.
#
# Point MODEL_REPO at a repo under YOUR Hugging Face account
# (the login cell above must have used a write token).
# ==========================================================
# ----------------------------------------------------------
# ENCODER INITIALISATION
# tum-nlp/bert-hateXplain was fine-tuned ON HateXplain, and 20,109 HateXplain
# rows are in our corpus - including part of our held-out test split. Starting
# from it therefore contaminates the in-distribution score that the whole
# model-selection comparison rests on. bert-base-uncased has no exposure to any
# text we evaluate on. The off-the-shelf hateXplain checkpoint is still
# evaluated, as an EXTERNAL baseline, in the BASELINES cell.
# ----------------------------------------------------------
BASE_MODEL = "roberta-base"   # RoBERTa replication: only this line and the two above change

FORCE_RETRAIN = True     # MUST stay True for every ablation run
USE_CONTRASTIVE = _RUN_CFG[RUN_LABEL]["contrastive"]   # driven by the RUN SELECTOR above
MODEL_REPO = "no-push/roberta-replication"   # Hub pushes are disabled in this notebook   # <-- change to your HF repo id
LOCAL_MODEL_DIR = "best_model"

import os
from pathlib import Path

MODEL_CACHED = False
if not FORCE_RETRAIN:
    if Path(LOCAL_MODEL_DIR, "config.json").exists():
        MODEL_CACHED = True
        print(f"Found local ./{LOCAL_MODEL_DIR} - reusing it, no training needed.")
    else:
        try:
            from huggingface_hub import snapshot_download
            snapshot_download(repo_id=MODEL_REPO, local_dir=LOCAL_MODEL_DIR)
            MODEL_CACHED = Path(LOCAL_MODEL_DIR, "config.json").exists()
            if MODEL_CACHED:
                print(f"Pulled fine-tuned model from HF Hub: {MODEL_REPO}")
        except Exception as e:
            print(f"No cached model on the Hub ({e!r}) - will train from scratch.")

print("MODEL_CACHED =", MODEL_CACHED, "| FORCE_RETRAIN =", FORCE_RETRAIN)

import glob as _glob

def _kaggle_input(filename, *fallbacks):
    """Locate a data file under /kaggle/input regardless of the mount path.
    Kaggle mounts a dataset at /kaggle/input/<slug>/, NOT /kaggle/input/datasets/<user>/<slug>/."""
    for cand in (_glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
                 + list(fallbacks) + [filename]):
        if os.path.exists(cand):
            return cand
    return filename


SPLIT_SEED = 42
SPLIT_HOLDOUT_FRAC = 0.20   # dev + test together; split 50/50 -> 80 / 10 / 10

def load_eval_split(which):
    """'dev' or 'test' from the 3-way stratified split. Prefers the CSVs the
    training cell writes; otherwise reconstructs them deterministically."""
    import pandas as _pd
    _p = f"/kaggle/working/{which}_split.csv"
    if os.path.exists(_p):
        return _pd.read_csv(_p)
    from sklearn.model_selection import train_test_split as _tts
    _d = _pd.read_csv("/kaggle/working/augmented_dataset.csv").dropna(subset=["text", "clean_label"])
    _d["clean_label"] = _d["clean_label"].astype(int)
    _tr, _h = _tts(_d, test_size=SPLIT_HOLDOUT_FRAC, random_state=SPLIT_SEED, stratify=_d["clean_label"])
    _dev, _te = _tts(_h, test_size=0.50, random_state=SPLIT_SEED, stratify=_h["clean_label"])
    return _dev if which == "dev" else _te


# RUN_LABEL is set in the RUN SELECTOR block at the top of this cell.
RUN_METRICS_PATH = "/kaggle/working/run_metrics.json"   # keyed by run_id = RUN_LABEL_sSEED
try:
    os.remove(RUN_METRICS_PATH)  # fresh row per run
except FileNotFoundError:
    pass

def record_metrics(**kw):
    """Merge results into run_metrics.json (the ablation ledger reads this)."""
    import json as _j
    m = {}
    if os.path.exists(RUN_METRICS_PATH):
        try: m = _j.load(open(RUN_METRICS_PATH))
        except Exception: m = {}
    for k, v in kw.items():
        try: m[k] = float(v)
        except (TypeError, ValueError): m[k] = v
    _j.dump(m, open(RUN_METRICS_PATH, "w"), indent=2)
    return m

record_metrics(run_label=RUN_LABEL,
               soft_label_training=int(SOFT_LABEL_TRAINING),
               soft_label_alpha=SOFT_LABEL_ALPHA,
               soft_disagreement_upweight=SOFT_DISAGREEMENT_UPWEIGHT,
               use_contrastive=int(USE_CONTRASTIVE),
               base_model=BASE_MODEL,
               train_seed=TRAIN_SEED,
               split_seed=SPLIT_SEED,
               run_id=f"{RUN_LABEL}_s{TRAIN_SEED}")

print(f"RUN_LABEL={RUN_LABEL} | TRAIN_SEED={TRAIN_SEED} | SOFT={SOFT_LABEL_TRAINING} | "
      f"CONTRASTIVE={USE_CONTRASTIVE} | UPWEIGHT={SOFT_DISAGREEMENT_UPWEIGHT} | "
      f"BASE={BASE_MODEL} | SPLIT_SEED={SPLIT_SEED} | FORCE_RETRAIN={FORCE_RETRAIN}")


In [ ]:
# --- cache guard: skip the dataset rebuild when a trained model is already available ---
if MODEL_CACHED:
    raise SystemExit("MODEL_CACHED -> skipping dataset augmentation (set FORCE_RETRAIN=True to rebuild).")
if not FORCE_RETRAIN and os.path.exists("/kaggle/working/augmented_dataset.csv"):
    raise SystemExit("augmented_dataset.csv already present -> skipping rebuild (FORCE_RETRAIN=True to force).")

# ==========================================================
# AUGMENT DATASET: hateXplain + ToxiGen + Latent Hatred + Multi-Sarcasm
# Run this in Colab/Kaggle (needs internet + `pip install datasets`)
# ==========================================================

# pip install datasets   # uncomment if not already installed

import pandas as pd
import numpy as np
from datasets import load_dataset

LABELS = {"hate speech": 0, "offensive": 1, "normal": 2}

# ------------------------------------------------------------------
# 0. Load your existing hateXplain CSV (the base dataset you already have)
# ------------------------------------------------------------------
# ---------------------------------------------------------------------------
# SAME-SOURCE ABLATION FIX
# Both the soft-label and hard-label arms are now built from the SAME HateXplain
# source (the annotator-level JSON). The hard arm simply collapses the 3-annotator
# distribution to its majority label. Previously the hard arm read a separate
# hateexplain.csv, so `soft_off` differed in BOTH loss and corpus (5,673 vs 6,306
# test rows) and the macro-F1 gap was not attributable to the loss alone.
# Set HX_SAME_SOURCE = False only to reproduce the old, confounded behaviour.
# ---------------------------------------------------------------------------
HX_SAME_SOURCE = True

if SOFT_LABEL_TRAINING or HX_SAME_SOURCE:
    # HateXplain WITH per-annotator labels: train on the empirical 3-annotator
    # label distribution instead of a single aggregated label -> the probability
    # bars then model disagreement, not just a point estimate.
    _HX_STR = {"hatespeech": 0, "normal": 1, "offensive": 2}   # HateXplain's own order
    HX_MAP  = {0: 0, 1: 2, 2: 1}                               # -> ours (0 hate, 1 offensive, 2 normal)

    def _hx_idx(lab):
        return int(lab) if isinstance(lab, (int, np.integer)) else _HX_STR[lab]

    def _load_hatexplain_annotated():
        # NOTE: both Hub attempts below are EXPECTED to fail on current `datasets`
        # versions - HF removed support for script-based datasets, and hatexplain
        # ships a hatexplain.py loader. The printed RuntimeError /
        # DatasetNotFoundError lines are informational, NOT a failure: the raw
        # GitHub JSON fallback immediately after is the working path and is what
        # produces the ~20k soft-labelled rows.
        for _kw in (dict(path="hatexplain", trust_remote_code=True),
                    dict(path="Paul/hatexplain", trust_remote_code=True)):
            try:
                _ds = load_dataset(**_kw)
                return [ex for sp in _ds for ex in _ds[sp]]
            except Exception as e:
                print("  ", _kw["path"], "unavailable:", repr(e)[:120])
        import requests
        _url = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/dataset.json"
        _raw = requests.get(_url, timeout=60).json()
        return [{"post_tokens": r["post_tokens"],
                 "annotators": {"label": [a["label"] for a in r["annotators"]]}}
                for r in _raw.values()]

    _rows = []
    for _ex in _load_hatexplain_annotated():
        _v = np.zeros(3)
        for _lab in _ex["annotators"]["label"]:
            _v[HX_MAP[_hx_idx(_lab)]] += 1
        _v = _v / _v.sum()
        _rows.append((" ".join(_ex["post_tokens"]), int(_v.argmax()), _v[0], _v[1], _v[2]))
    base_df = (pd.DataFrame(_rows, columns=["text", "clean_label",
                                            "soft_hate", "soft_offensive", "soft_normal"])
                 .drop_duplicates(subset=["text"]).reset_index(drop=True))
    base_df["source"] = "hateXplain"
    _dis = int((base_df[["soft_hate", "soft_offensive", "soft_normal"]].max(axis=1) < 0.999).sum())
    print(f"Base HateXplain rows (with soft labels): {len(base_df)}  |  annotator-disagreement rows: {_dis}")

    if not SOFT_LABEL_TRAINING:
        # HARD-LABEL ARM, SAME SOURCE: keep clean_label (already the annotator
        # majority via argmax) and drop the soft columns so the training cell
        # falls back to plain cross-entropy. Identical rows, identical split,
        # only the loss differs.
        base_df = base_df.drop(columns=["soft_hate", "soft_offensive", "soft_normal"])
        print(f"  HARD-LABEL arm: soft columns dropped, {len(base_df)} rows kept "
              f"(same source as the soft arm - loss is the only difference)")
elif False:
    base_df = pd.read_csv(_kaggle_input("hateexplain.csv",
        "/kaggle/input/datasets/alinashifa/hateexplain-csv/hateexplain.csv"))
    base_df["clean_label"] = base_df["label"].str.lower().str.strip().map(LABELS)
    base_df = base_df.dropna(subset=["clean_label"]).reset_index(drop=True)
    base_df = base_df[["comment", "clean_label"]].rename(columns={"comment": "text"})
    base_df["source"] = "hateXplain"
    print("Base hateXplain rows:", len(base_df))

# ------------------------------------------------------------------
# 1. ToxiGen — implicit toxicity, no slurs, mentions minority groups
#    Fields: 'generation' (text), 'prompt_label' (1=toxic, 0=benign)
# ------------------------------------------------------------------
toxigen = load_dataset("toxigen/toxigen-data", "train")["train"]
print("ToxiGen columns:", toxigen.column_names)  # sanity check — adjust below if names differ

toxigen_df = toxigen.to_pandas()
text_col = "generation" if "generation" in toxigen_df.columns else "text"
label_col = "prompt_label" if "prompt_label" in toxigen_df.columns else "label"

toxigen_df = toxigen_df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "tox_label"})
# prompt_label labels the PROMPT, not the generation: it agrees with human judgment for
# only ~68% of top-k toxic examples (~40% ALICE-decoded) but ~92-95% for benign. So the
# machine set contributes ONLY prompt_label == 0 -> normal (identity-mention FP traps);
# the toxic side comes entirely from the human-annotated ToxiGen_human subset below.
toxigen_df = toxigen_df[toxigen_df["tox_label"] == 0].reset_index(drop=True)
toxigen_df["clean_label"] = LABELS["normal"]
toxigen_df["source"] = "ToxiGen_machine_benign"
toxigen_df = toxigen_df[["text", "clean_label", "source"]]
# These are identity-mention false-positive traps, not the bulk of training data. Keeping
# all ~39k made the corpus 40% machine-benign and turned the model into a normal-detector
# (0 benign FPs, ~27% bullying recall). Subsample to ~human-subset scale.
TOXIGEN_MACHINE_BENIGN_N = 6000
toxigen_df = toxigen_df.sample(n=min(TOXIGEN_MACHINE_BENIGN_N, len(toxigen_df)),
                               random_state=42).reset_index(drop=True)
print(f"ToxiGen benign (prompt_label==0) rows kept: {len(toxigen_df)} "
      f"(capped at TOXIGEN_MACHINE_BENIGN_N={TOXIGEN_MACHINE_BENIGN_N})")

# 1b. ToxiGen HUMAN-annotated subset. prompt_label labels the *prompt*, not the
#     generation; the "annotated" config carries per-text human toxicity scores.
# 1-5 Likert mean across annotators. 3.0 is precisely where annotators split, so a
# single midpoint threshold forces the ambiguous middle into "toxic" and dumps mildly
# toxic 2.x rows into "normal". Two thresholds, middle band dropped, instead.
TOXIGEN_TOXIC_THRESH  = 4.0
TOXIGEN_BENIGN_THRESH = 2.0
try:
    _tga = load_dataset("toxigen/toxigen-data", "annotated")
    _tga = pd.concat([_tga[s].to_pandas() for s in _tga], ignore_index=True)
    if "toxicity_human" not in _tga.columns:
        raise KeyError(f"toxicity_human missing; got {list(_tga.columns)} - refusing to fall "
                       f"back to toxicity_ai (that IS the machine label this fix removes)")
    _tcol = "text" if "text" in _tga.columns else "generation"
    toxigen_human_df = _tga[[_tcol, "toxicity_human"]].rename(
        columns={_tcol: "text", "toxicity_human": "_th"})
    toxigen_human_df["_th"] = pd.to_numeric(toxigen_human_df["_th"], errors="coerce")
    toxigen_human_df = toxigen_human_df.dropna(subset=["_th"]).reset_index(drop=True)
    _pre = len(toxigen_human_df)
    _keep = ((toxigen_human_df["_th"] >= TOXIGEN_TOXIC_THRESH)
             | (toxigen_human_df["_th"] <= TOXIGEN_BENIGN_THRESH))
    toxigen_human_df = toxigen_human_df[_keep].reset_index(drop=True)
    toxigen_human_df["clean_label"] = np.where(
        toxigen_human_df["_th"] >= TOXIGEN_TOXIC_THRESH, LABELS["hate speech"], LABELS["normal"])
    toxigen_human_df["source"] = "ToxiGen_human"
    toxigen_human_df = toxigen_human_df[["text", "clean_label", "source"]].drop_duplicates("text")
    toxigen_df = toxigen_df[~toxigen_df["text"].isin(set(toxigen_human_df["text"]))].reset_index(drop=True)
    print(f"ToxiGen human rows: {len(toxigen_human_df)} kept (>= {TOXIGEN_TOXIC_THRESH} toxic, "
          f"<= {TOXIGEN_BENIGN_THRESH} benign) | {_pre - len(toxigen_human_df)} middle-band rows "
          f"dropped | machine-benign rows after dedup: {len(toxigen_df)}")
except Exception as _e:
    print("ToxiGen annotated subset unavailable:", repr(_e)[:150])
    toxigen_human_df = pd.DataFrame(columns=["text", "clean_label", "source"])

# ------------------------------------------------------------------
# 2. Latent Hatred — real tweets with coded/implicit hate language
#    Fields: 'post' (text), 'class' (implicit_hate / explicit_hate / not_hate)
# ------------------------------------------------------------------
latent = load_dataset("tasksource/implicit-hate-stg1")["train"]
print("Latent Hatred columns:", latent.column_names)  # sanity check

latent_df = latent.to_pandas()
text_col = "post" if "post" in latent_df.columns else "text"
label_col = "class" if "class" in latent_df.columns else "label"

latent_df = latent_df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "lh_label"})
latent_label_map = {
    "implicit_hate": LABELS["hate speech"],
    "explicit_hate": LABELS["hate speech"],
    "not_hate": LABELS["normal"],
}
latent_df["clean_label"] = latent_df["lh_label"].map(latent_label_map)
latent_df = latent_df.dropna(subset=["clean_label"]).reset_index(drop=True)
latent_df["source"] = "LatentHatred"
latent_df = latent_df[["text", "clean_label", "source"]]
print("Latent Hatred rows:", len(latent_df))

# ------------------------------------------------------------------
# 3. Multi-Sarcasm -- DROPPED from training. The sarcasm_level -> toxicity tertile
#    mapping is an unvalidated labelling assumption; see the sarcasm-audit cell,
#    which keeps the 200-row audit sample as a released artifact.

# ------------------------------------------------------------------
# 3.5. GoEmotions — genuine positive/warm sentiment, to teach the model
#      what real appreciation/kindness looks like, not just "not flagged"
#      Fields: 'text', 'labels' (list of emotion label IDs)
# ------------------------------------------------------------------
goemotions = load_dataset("google-research-datasets/go_emotions", "simplified")["train"]
print("GoEmotions columns:", goemotions.column_names)

ge_label_names = goemotions.features["labels"].feature.names
print("GoEmotions label names sample:", ge_label_names[:10])

# Emotions that represent genuine warmth/appreciation -> reinforce as "normal"
POSITIVE_EMOTIONS = {"admiration", "gratitude", "joy", "love", "caring", "optimism", "approval", "pride"}
positive_label_ids = {i for i, name in enumerate(ge_label_names) if name in POSITIVE_EMOTIONS}

ge_df = goemotions.to_pandas()
ge_df["has_positive_emotion"] = ge_df["labels"].apply(
    lambda lbls: any(l in positive_label_ids for l in lbls)
)
ge_positive_df = ge_df[ge_df["has_positive_emotion"]][["text"]].copy()
ge_positive_df["clean_label"] = LABELS["normal"]
ge_positive_df["source"] = "GoEmotions_positive"

# Subsample to a reasonable size — this is reinforcement, not the bulk of training data
ge_positive_df = ge_positive_df.sample(
    n=min(8000, len(ge_positive_df)), random_state=42
).reset_index(drop=True)
print("GoEmotions positive rows:", len(ge_positive_df))
print("ASSUMPTION (unaudited): GoEmotions positive-emotion labels (admiration/gratitude/"
      "joy/love/caring/optimism/approval/pride) -> normal. Lower risk than the sarcasm "
      "mapping, but approval/optimism can co-occur with sarcasm; not spot-checked.")

# ------------------------------------------------------------------
# 3.6. Hand-written contrastive examples (meta-commentary vs. attack,
#      backhanded vs. genuine compliment, exclusion/concern/gaslighting
#      vs. their benign counterparts). Small, but targets specific
#      decision boundaries the bulk data doesn't cover.
#      Files expected to have columns: text, clean_label, category, rationale
# ------------------------------------------------------------------
contrastive_paths = [
    _kaggle_input("contrastive_examples_final.csv",
                  "/kaggle/input/datasets/alinashifa/contrastive-final/contrastive_examples_final.csv"),
]

PAIRED_CATEGORIES = [
    ("meta_commentary",            "direct_attack"),
    ("meta_commentary",            "direct_attack_same_topic"),
    ("untargeted_absurdist_humor", "targeted_attack"),
    ("genuine_warmth",             "sarcastic_insult"),
    ("genuine_compliment",         "backhanded_compliment"),
    ("genuine_compliment",         "sarcastic_insult"),
    ("neutral_statement",          "dogwhistle"),
    ("dogwhistle",                 "coded_offensive"),
    ("dark_humor_benign",          "dark_humor_targeted"),
    ("genuine_criticism",          "dismissive_insult"),
]


def _add_pair_ids(dfc):
    """pair_id for genuinely paired contrastive categories. Uses a (category, key)
    column pair if present, else falls back to ADJACENCY: consecutive rows whose
    categories are a known contrastive sibling pair get the same id. Unpaired rows
    stay blank."""
    if "pair_id" in dfc.columns:
        return dfc
    dfc = dfc.reset_index(drop=True).copy()
    dfc["pair_id"] = ""
    _cat = next((c for c in ("category", "cat", "type") if c in dfc.columns), None)
    _key = next((c for c in ("topic", "anchor", "pair", "ref", "ref_id", "pair_key") if c in dfc.columns), None)
    # a category can have more than one sibling (meta_commentary pairs with two attack
    # categories, genuine_compliment with two insult categories) -> frozensets, not a dict.
    _sib = {frozenset(p) for p in PAIRED_CATEGORIES}
    _flat = {c for pr in _sib for c in pr}
    if _cat and _key:
        _n = 0
        for _k, _g in dfc[dfc[_cat].astype(str).isin(_flat)].groupby(_key):
            if _g[_cat].nunique() >= 2:
                _n += 1
                dfc.loc[_g.index, "pair_id"] = f"P{_n:03d}"
        print(f"  pair_id: {_n} pairs from ({_cat}, {_key})")
    elif _cat:
        _cats = dfc[_cat].astype(str).tolist()
        _n, i = 0, 0
        while i < len(_cats) - 1:
            if frozenset((_cats[i], _cats[i + 1])) in _sib and not dfc.at[i, "pair_id"]:
                _n += 1
                dfc.at[i, "pair_id"] = dfc.at[i + 1, "pair_id"] = f"P{_n:03d}"
                i += 2
            else:
                i += 1
        _blank = int((dfc["pair_id"] == "").sum())
        print(f"  pair_id: {_n} pairs by adjacency on ({_cat}); {_blank} rows unpaired "
              f"(add a `topic` column by hand if this is 0)")
        if _n < 20:
            print(f"  !!! WARNING: only {_n} contrastive pairs found - check PAIRED_CATEGORIES "
                  f"against the actual category values: {sorted(set(_cats))}")
    else:
        print("  pair_id: left blank - no category column")
    return dfc


contrastive_dfs = []
if True:   # always load: USE_CONTRASTIVE is applied at TRAIN time, not corpus time
    for p in contrastive_paths:
        if not os.path.exists(p):
            print(f"Skipping {p} - not found")
            continue
        cdf_full = _add_pair_ids(pd.read_csv(p))
        cdf_full.to_csv("/kaggle/working/contrastive_released.csv", index=False)
        try:
            from huggingface_hub import upload_file
            raise RuntimeError("Hub upload disabled")
            upload_file(path_or_fileobj="/kaggle/working/contrastive_released.csv",
                        path_in_repo="contrastive_examples_final.csv", repo_id=MODEL_REPO)
        except Exception:
            pass
        cdf = cdf_full[["text", "clean_label"]].copy()
        cdf["clean_label"] = cdf["clean_label"].astype(int)
        cdf["source"] = "Contrastive_handwritten"
        contrastive_dfs.append(cdf)
        print(f"Loaded {p}: {len(cdf)} rows")
        break
    if not contrastive_dfs:
        print("!!! WARNING: USE_CONTRASTIVE=True but NO contrastive CSV was found - "
              "`main` and `no_contrastive` will be identical. Check the dataset slug / "
              "glob /kaggle/input/**/contrastive_examples_final.csv")
else:
    print("contrastive loader skipped")

if contrastive_dfs:
    contrastive_df = pd.concat(contrastive_dfs, ignore_index=True)
else:
    contrastive_df = pd.DataFrame(columns=["text", "clean_label", "source"])

print("Total contrastive rows:", len(contrastive_df))
if "record_metrics" in dir():
    record_metrics(contrastive_rows=len(contrastive_df),
                   contrastive_found=int(len(contrastive_dfs) > 0))

# ------------------------------------------------------------------
# 4. Merge everything into one augmented dataset
# ------------------------------------------------------------------
base_df["clean_label"] = base_df["clean_label"].astype(int)
toxigen_df["clean_label"] = toxigen_df["clean_label"].astype(int)
latent_df["clean_label"] = latent_df["clean_label"].astype(int)
ge_positive_df["clean_label"] = ge_positive_df["clean_label"].astype(int)

# label provenance: human annotation vs model-generated vs heuristic mapping
_PROV = {"hateXplain": "human", "ToxiGen": "machine", "ToxiGen_machine": "machine",
         "ToxiGen_machine_benign": "machine",
         "ToxiGen_human": "human", "LatentHatred": "human",
         "GoEmotions_positive": "heuristic"}
for _d in (base_df, toxigen_df, toxigen_human_df, latent_df, ge_positive_df, contrastive_df):
    if len(_d):
        _d["label_source"] = _d["source"].map(
            lambda s: _PROV.get(s, "human" if str(s).startswith("Contrastive") else "heuristic"))

# MASTER-SPLIT FIX : the contrastive rows are ALWAYS loaded and
# ALWAYS merged, so the corpus - and therefore the stratified split - is byte
# identical for every arm. USE_CONTRASTIVE no longer changes the corpus; it is
# applied later, in the training cell, by dropping the tagged rows from the TRAIN
# partition only. Dev and test are then identical across all arms and all seeds.
augmented_df = pd.concat(
    [base_df, toxigen_df, toxigen_human_df, latent_df, ge_positive_df, contrastive_df],
    ignore_index=True,
)
augmented_df["label_source"] = augmented_df["label_source"].fillna("heuristic")
augmented_df = augmented_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# soft targets: real annotator distributions for HateXplain, one-hot for every other source
if SOFT_LABEL_TRAINING:
    _sc = ["soft_hate", "soft_offensive", "soft_normal"]
    for _c in _sc:
        if _c not in augmented_df.columns:
            augmented_df[_c] = np.nan
    _need = augmented_df[_sc].isna().any(axis=1)
    augmented_df.loc[_need, _sc] = np.eye(3)[augmented_df.loc[_need, "clean_label"].astype(int).to_numpy()]
    print(f"Soft targets: {(~_need).sum()} annotator-derived, {int(_need.sum())} one-hot")


print("\n=== Final dataset summary ===")
print("Total rows:", len(augmented_df))
print(augmented_df["source"].value_counts())
print(augmented_df["label_source"].value_counts())
print(augmented_df["clean_label"].value_counts().rename(index={v: k for k, v in LABELS.items()}))

print("\nsource x label crosstab (class 'offensive' provenance concentration):")
print(pd.crosstab(augmented_df["source"],
                  augmented_df["clean_label"].map({0: "hate", 1: "offensive", 2: "normal"})))
print("LIMITATION: with MultiSarcasm dropped, ToxiGen/LatentHatred/GoEmotions only ever "
      "emit hate or normal - class 'offensive' comes almost entirely from hateXplain and "
      "the small contrastive set. This concentration is a real constraint on the 3-class "
      "model and will show up as weaker offensive-class F1/recall.")
if "record_metrics" in dir():
    _off_src = augmented_df.loc[augmented_df["clean_label"] == 1, "source"].value_counts()
    record_metrics(
        offensive_class_top_source=(str(_off_src.index[0]) if len(_off_src) else "none"),
        offensive_class_top_source_frac=(float(_off_src.iloc[0] / max(1, _off_src.sum()))
                                        if len(_off_src) else 0.0),
        toxigen_policy="human_toxic_machine_benign",
        toxigen_toxic_thresh=TOXIGEN_TOXIC_THRESH,
        toxigen_benign_thresh=TOXIGEN_BENIGN_THRESH)

augmented_df.to_csv("augmented_dataset.csv", index=False)
print("\nSaved -> augmented_dataset.csv")


In [ ]:
# ============================================================
# SARCASM MAPPING AUDIT  -  RETIRED (kept for provenance only)
# ============================================================
# MultiSarcasm was dropped from the dataset entirely after the audit sample showed
# the sarcasm_level tertile mapping labelled ordinary comments ("Verified :)",
# "Yay supply drops for MW2 Remastered") as OFFENSIVE. Nothing downstream reads
# from it, so there is nothing left to audit and no CSV to fill in.
#
# The 200-row audit sample is kept as the record of the drop decision.
# Re-enable only if MultiSarcasm is ever
# reintroduced.
# ------------------------------------------------------------
print("Sarcasm audit: RETIRED - MultiSarcasm is not used for training. "
      "No labelling required, nothing to run here.")


In [ ]:
# --- cache guard: skip training when a fine-tuned model is already available ---
if MODEL_CACHED:
    raise SystemExit(f"MODEL_CACHED -> using ./{LOCAL_MODEL_DIR}, skipping training (FORCE_RETRAIN=True to retrain).")

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# ---- reproducibility: SPLIT_SEED governs the partition (fixed for all arms),
# ---- TRAIN_SEED governs training stochasticity (varied 0/1/2).
import random as _rnd
SPLIT_SEED  = int(globals().get("SPLIT_SEED", 42))
TRAIN_SEED  = int(globals().get("TRAIN_SEED", 0))
_rnd.seed(TRAIN_SEED); np.random.seed(TRAIN_SEED); torch.manual_seed(TRAIN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TRAIN_SEED)
try:
    from transformers import set_seed as _hf_set_seed; _hf_set_seed(TRAIN_SEED)
except Exception:
    pass
print(f"seeds -> SPLIT_SEED={SPLIT_SEED} (fixed)  TRAIN_SEED={TRAIN_SEED} (varied)")

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
def _f1s_full(y, p):
    from sklearn.metrics import f1_score as _f
    return _f(y, p, average='macro', labels=[0, 1, 2], zero_division=0)
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
from torch.optim import AdamW

# ------------------------------------------------------------------
# 1. LOAD THE AUGMENTED DATASET
# ------------------------------------------------------------------
csv_path = "/kaggle/working/augmented_dataset.csv"  # HateXplain(soft)+ToxiGen(human-toxic/machine-benign)+LatentHatred+GoEmotions(+contrastive if present)
df = pd.read_csv(csv_path)
print("Columns:", df.columns.tolist())
print(df["clean_label"].value_counts())

LABELS = {"hate speech": 0, "offensive": 1, "normal": 2}
LABELS_INV = {v: k for k, v in LABELS.items()}

df = df.dropna(subset=["text", "clean_label"]).reset_index(drop=True)
df["clean_label"] = df["clean_label"].astype(int)

# --- soft-label setup (from the CACHE CONFIG cell) --------------------------
SOFT_LABEL_TRAINING = bool(globals().get("SOFT_LABEL_TRAINING", False))
SOFT_LABEL_ALPHA    = float(globals().get("SOFT_LABEL_ALPHA", 1.0))
SOFT_DISAGREEMENT_UPWEIGHT = float(globals().get("SOFT_DISAGREEMENT_UPWEIGHT", 1.0))
SOFT_COLS = ["soft_hate", "soft_offensive", "soft_normal"]
USE_SOFT  = SOFT_LABEL_TRAINING and all(c in df.columns for c in SOFT_COLS)
if USE_SOFT:
    df[SOFT_COLS] = df[SOFT_COLS].fillna(0.0).astype(float)
    _rs = df[SOFT_COLS].sum(axis=1).replace(0, 1.0)
    df[SOFT_COLS] = df[SOFT_COLS].div(_rs, axis=0)          # renormalise defensively
    _ndis = int((df[SOFT_COLS].max(axis=1) < 0.999).sum())
    print(f"Soft-label training ON (alpha={SOFT_LABEL_ALPHA}) | {_ndis} rows carry annotator disagreement")
else:
    print("Soft-label training OFF -> hard-label class-weighted cross-entropy")

# ------------------------------------------------------------------
# 2. TRAIN / VALIDATION SPLIT (stratified so class balance is preserved)
# ------------------------------------------------------------------
train_df, _hold = train_test_split(
    df, test_size=0.20, random_state=SPLIT_SEED, stratify=df["clean_label"]
)
val_df, test_df = train_test_split(          # val_df is the DEV set: checkpoint / temperature / threshold
    _hold, test_size=0.50, random_state=SPLIT_SEED, stratify=_hold["clean_label"]
)
# ---- MASTER-SPLIT ABLATION: apply USE_CONTRASTIVE to TRAIN only ----
# The corpus, and therefore the split, is identical for every arm. The
# no_contrastive ablation removes the 155 hand-written rows from the TRAINING
# partition after the split, so dev and test are byte-identical across all arms
# and all seeds. Any contrastive rows that landed in dev/test are removed from
# BOTH evaluation partitions in every arm, so no author-written text is ever
# scored and the evaluation sets stay identical.
_is_contr = lambda d: d["source"].astype(str).str.startswith("Contrastive")
_n_dev_c, _n_test_c = int(_is_contr(val_df).sum()), int(_is_contr(test_df).sum())
val_df  = val_df[~_is_contr(val_df)].reset_index(drop=True)
test_df = test_df[~_is_contr(test_df)].reset_index(drop=True)
print(f"  removed {_n_dev_c} dev / {_n_test_c} test contrastive rows "
      f"(done in EVERY arm -> identical eval sets)")
if not USE_CONTRASTIVE:
    _n0 = len(train_df)
    train_df = train_df[~_is_contr(train_df)].reset_index(drop=True)
    print(f"  USE_CONTRASTIVE=False -> dropped {_n0-len(train_df)} contrastive rows "
          f"from TRAIN only ({_n0} -> {len(train_df)})")
else:
    print(f"  USE_CONTRASTIVE=True  -> {int(_is_contr(train_df).sum())} contrastive rows kept in TRAIN")

val_df.to_csv("/kaggle/working/dev_split.csv", index=False)
test_df.to_csv("/kaggle/working/test_split.csv", index=False)
print(f"Train {len(train_df)} | Dev {len(val_df)} | Test {len(test_df)}   (test is scored once, at the very end)")
if "record_metrics" in dir():
    record_metrics(n_train=len(train_df), n_dev=len(val_df), n_test=len(test_df))

# ------------------------------------------------------------------
# 3. MODEL + TOKENIZER SETUP
# ------------------------------------------------------------------
model_checkpoint = globals().get("BASE_MODEL", "bert-base-uncased")
# NOTE: previously tum-nlp/bert-hateXplain. Changed to a clean base because that
# checkpoint was fine-tuned on HateXplain, which supplies 20,109 rows of this
# corpus including part of the held-out test split. Expect lower absolute
# scores; what matters is whether the
# ordering between configurations survives.
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config = AutoConfig.from_pretrained(model_checkpoint, num_labels=3, output_attentions=True)
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, config=config, ignore_mismatched_sizes=True, torch_dtype=torch.float32
)
model = model.float()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ------------------------------------------------------------------
# 4. TOKENIZE (longer max_length to fit Reddit/ToxiGen sentences)
# ------------------------------------------------------------------
MAX_LEN = 128

print("Tokenizing train set...")
train_encodings = tokenizer(
    list(train_df["text"]), truncation=True, padding=True, max_length=MAX_LEN
)
print("Tokenizing val set...")
val_encodings = tokenizer(
    list(val_df["text"]), truncation=True, padding=True, max_length=MAX_LEN
)
test_encodings = tokenizer(
    list(test_df["text"]), truncation=True, padding=True, max_length=MAX_LEN
)


class ToxicityDataset(Dataset):
    def __init__(self, encodings, labels, soft=None):
        self.encodings = encodings
        self.labels = labels
        self.soft = soft

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]))
        if self.soft is not None:
            item["soft_labels"] = torch.tensor(self.soft[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)


_train_soft = train_df[SOFT_COLS].to_numpy() if USE_SOFT else None
_val_soft   = val_df[SOFT_COLS].to_numpy() if USE_SOFT else None
train_dataset = ToxicityDataset(train_encodings, list(train_df["clean_label"]), _train_soft)
val_dataset = ToxicityDataset(val_encodings, list(val_df["clean_label"]), _val_soft)

_g = torch.Generator(); _g.manual_seed(TRAIN_SEED)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, generator=_g)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataset = ToxicityDataset(test_encodings, list(test_df["clean_label"]),
                               test_df[SOFT_COLS].to_numpy() if USE_SOFT else None)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ------------------------------------------------------------------
# 5. CLASS WEIGHTS (so the majority class doesn't dominate the loss)
# ------------------------------------------------------------------
class_weights_raw = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["clean_label"].values,
)
# Milder dampening than sqrt (0.5) — full "balanced" weighting caused
# over-flagging of benign text; sqrt swung too far the other way and
# caused under-flagging of real toxic/coded content. This run uses 0.5 (sqrt
# dampening); 0.75 is the other value tried - keep whichever the ablation supports.
DAMPEN_EXPONENT = 0.5
class_weights = class_weights_raw ** DAMPEN_EXPONENT
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Raw class weights:", class_weights_raw)
print(f"Dampened class weights (^{DAMPEN_EXPONENT}):", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
_logsoftmax = nn.LogSoftmax(dim=-1)

def compute_loss(logits, labels, soft=None):
    """Class-weighted hard CE, or class-weighted SOFT cross-entropy when annotator
    distributions are available. Rows whose annotators disagreed are upweighted by
    SOFT_DISAGREEMENT_UPWEIGHT so they are not drowned out by the one-hot rows.
    A one-hot soft target reduces exactly to weighted CE."""
    hard = criterion(logits, labels)
    if soft is None:
        return hard
    logp = _logsoftmax(logits)
    per_ex = -(class_weights.unsqueeze(0) * soft * logp).sum(dim=1)            # (B,)
    w = torch.where(soft.max(dim=1).values < 0.999,
                    per_ex.new_tensor(SOFT_DISAGREEMENT_UPWEIGHT),
                    per_ex.new_tensor(1.0))
    soft_ce = (per_ex * w).sum() / w.sum()
    return SOFT_LABEL_ALPHA * soft_ce + (1.0 - SOFT_LABEL_ALPHA) * hard
optim = AdamW(model.parameters(), lr=2e-5)

# ------------------------------------------------------------------
# 6. EVALUATION FUNCTION
# ------------------------------------------------------------------
def evaluate_disagreement(model, loader):
    """On val rows where annotators actually disagreed: how close is the model's
    predictive distribution to theirs, and does model entropy track human entropy?
    Diagnostic only - checkpoint selection stays on macro-F1."""
    model.eval()
    kl, h_human, h_model = [], [], []
    with torch.no_grad():
        for batch in loader:
            if "soft_labels" not in batch:
                return None
            q = batch["soft_labels"]
            nd = q.max(dim=1).values < 0.999
            if nd.sum() == 0:
                continue
            logits = model(input_ids=batch["input_ids"].to(device),
                           attention_mask=batch["attention_mask"].to(device)).logits.cpu()
            p = logits.softmax(dim=1)
            qn, pn = q[nd].clamp_min(1e-9), p[nd].clamp_min(1e-9)
            kl.extend((qn * (qn.log() - pn.log())).sum(1).tolist())
            h_human.extend((-(qn * qn.log()).sum(1)).tolist())
            h_model.extend((-(pn * pn.log()).sum(1)).tolist())
    if not kl:
        return None
    r = float(np.corrcoef(h_human, h_model)[0, 1]) if len(h_human) > 2 else float("nan")
    return {"n": len(kl), "kl_q_p": float(np.mean(kl)), "entropy_corr": r}



def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    report = classification_report(
        all_labels, all_preds, target_names=["hate speech", "offensive", "normal"], digits=3
    )
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return report, macro_f1


# ------------------------------------------------------------------
# 7. TRAINING LOOP (with checkpointing on best val macro-F1)
# ------------------------------------------------------------------
EPOCHS = 8
PATIENCE = 3  # stop if val macro-F1 doesn't improve for this many consecutive epochs
best_f1 = 0.0
best_model_path = "best_model"
epochs_no_improve = 0
_spectrum_epochs = []   # (epoch, entropy_corr, KL) - reported together so it's not cherry-picked

BENIGN_CONTROL_SENTENCES = [
    "You are a good person.",
    "Hey! Take care of yourself.",
    "I really appreciate your help today.",
    "You did a great job on this project.",
    "Have a wonderful day!",
    "Thank you so much for being there for me.",
    "Congratulations on your achievement!",
]

def check_benign_false_positives(model, tokenizer):
    model.eval()
    flagged = 0
    for text in BENIGN_CONTROL_SENTENCES:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=1).item()
        if pred != 2:  # not "normal"
            flagged += 1
    return flagged, len(BENIGN_CONTROL_SENTENCES)

TOXIC_CONTROL_SENTENCES = [
    "Wow, you're SO smart, I'm amazed you found the door.",
    "Oh sure, because that worked out SO well last time.",
    "Nice job, genius.",
    "If you want your comment to get noticed, you post sexist, racist, homophobic etc content that other people won't be able to resist responding to.",
    "Some groups just aren't built for this kind of work.",
]

def check_toxic_recall(model, tokenizer):
    model.eval()
    caught = 0
    for text in TOXIC_CONTROL_SENTENCES:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=1).item()
        if pred != 2:  # correctly NOT "normal"
            caught += 1
    return caught, len(TOXIC_CONTROL_SENTENCES)

print("\n🚀 Training on augmented dataset...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_idx, batch in enumerate(train_loader):
        optim.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        soft = batch["soft_labels"].to(device) if "soft_labels" in batch else None
        loss = compute_loss(logits, labels, soft)
        loss.backward()
        optim.step()

        total_loss += loss.item()
        if batch_idx % 200 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} finished. Avg train loss: {avg_loss:.4f}")

    report, macro_f1 = evaluate(model, val_loader)
    print(f"Dev macro-F1: {macro_f1:.4f}")
    print(report)

    flagged, total_benign = check_benign_false_positives(model, tokenizer)
    print(f"⚠️  Benign control check: {flagged}/{total_benign} benign sentences misflagged as non-Normal")

    caught, total_toxic = check_toxic_recall(model, tokenizer)
    print(f"⚠️  Toxic control check: {caught}/{total_toxic} sarcastic/coded toxic sentences correctly caught")

    if USE_SOFT:
        _sp = evaluate_disagreement(model, val_loader)
        if _sp:
            _spectrum_epochs.append((epoch + 1, _sp["entropy_corr"], _sp["kl_q_p"]))
            print(f"🌈 Spectrum check (disagreement rows n={_sp['n']}): "
                  f"KL(annotators||model)={_sp['kl_q_p']:.3f}, "
                  f"entropy corr={_sp['entropy_corr']:.3f}")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        epochs_no_improve = 0
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        print(f"✅ New best model saved (macro-F1: {best_f1:.4f})")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s).")
        if epochs_no_improve >= PATIENCE:
            print(f"⏹ Early stopping triggered (no improvement for {PATIENCE} epochs).")
            break

print(f"\n🎉 Training complete. Best DEV macro-F1: {best_f1:.4f}")
print(f"Best model saved at ./{best_model_path}")

# ------------------------------------------------------------------
# 8. HELD-OUT TEST  -  loaded once, reported once, never a training signal
# ------------------------------------------------------------------
_best_for_test = AutoModelForSequenceClassification.from_pretrained(best_model_path).to(device).eval()
_test_report, _test_f1 = evaluate(_best_for_test, test_loader)
print("\n================  HELD-OUT TEST (reported once)  ================")
print(f"Test macro-F1: {_test_f1:.4f}")
print(_test_report)

# ---- bootstrap CIs over test items (single seed per run: CIs quantify item
# sampling error, NOT seed variance. Do not write "robust"/"stable" from these). ----
_test_preds, _test_gold = [], []
with torch.no_grad():
    for _b in test_loader:
        _test_preds.extend(_best_for_test(input_ids=_b["input_ids"].to(device),
                           attention_mask=_b["attention_mask"].to(device)).logits.argmax(1).cpu().tolist())
        _test_gold.extend(_b["labels"].cpu().tolist() if _b["labels"].dim() == 1
                          else _b["labels"].argmax(1).cpu().tolist())
_test_preds = np.asarray(_test_preds); _test_gold = np.asarray(_test_gold)

def _boot_metric(fn, gold, pred, n_boot=2000, seed=0):
    _rng = np.random.default_rng(seed); _m = len(gold); _ix = np.arange(_m)
    _s = np.empty(n_boot)
    for _k in range(n_boot):
        _r = _rng.choice(_ix, _m, replace=True)
        _s[_k] = fn(gold[_r], pred[_r])
    return float(np.percentile(_s, 2.5)), float(np.percentile(_s, 97.5))

from sklearn.metrics import f1_score as _f1b, recall_score as _recb
_ci_lo, _ci_hi = _boot_metric(lambda g, p: _f1b(g, p, average="macro", zero_division=0),
                              _test_gold, _test_preds)
print(f"\nTest macro-F1 95% CI (2000x item bootstrap): [{_ci_lo:.4f}, {_ci_hi:.4f}]")
_cls_ci = {}
for _ci_idx, _ci_name in [(0, "hate"), (1, "offensive"), (2, "normal")]:
    _lo, _hi = _boot_metric(lambda g, p, _i=_ci_idx: _recb(g == _i, p == _i, zero_division=0),
                            _test_gold, _test_preds)
    _cls_ci[_ci_name] = (_lo, _hi)
    print(f"  recall[{_ci_name:9s}] 95% CI [{_lo:.3f}, {_hi:.3f}]")
if "record_metrics" in dir():
    record_metrics(test_macro_f1_ci_lo=_ci_lo, test_macro_f1_ci_hi=_ci_hi,
                   **{f"test_recall_{k}_ci_lo": v[0] for k, v in _cls_ci.items()},
                   **{f"test_recall_{k}_ci_hi": v[1] for k, v in _cls_ci.items()})
if "label_source" in test_df.columns:
    from sklearn.metrics import recall_score as _rec_s
    _tp = []
    with torch.no_grad():
        for _b in test_loader:
            _tp.extend(_best_for_test(input_ids=_b["input_ids"].to(device),
                       attention_mask=_b["attention_mask"].to(device)).logits.argmax(1).cpu().tolist())
    _tv = test_df.reset_index(drop=True).assign(_pred=_tp)
    # macro-F1 is meaningless when a label_source subset has only one class present
    # (ToxiGen_machine_benign / GoEmotions_positive are all-normal); report accuracy and
    # per-present-class recall instead.
    print("\nTest by label_source (accuracy + per-present-class recall; macro-F1 only when >1 class):")
    _lm3 = {0: "hate", 1: "off", 2: "normal"}
    for _src, _g in _tv.groupby("label_source"):
        _present = sorted(_g["clean_label"].unique())
        _acc = float((_g["clean_label"] == _g["_pred"]).mean())
        _recs = {_lm3[c]: float(_rec_s(_g["clean_label"] == c, _g["_pred"] == c, zero_division=0))
                 for c in _present}
        _mf1 = ("" if len(_present) < 2 else
                f" macroF1={_f1s_full(_g['clean_label'], _g['_pred']):.3f}")
        print(f"  {_src:10s} n={len(_g):6d}  acc={_acc:.3f}  recall={_recs}{_mf1}")
if USE_SOFT:
    if _spectrum_epochs:
        print("\nSpectrum check, epoch-wise (entropy_corr wanders around 0 -> negative result):")
        for _e, _ec, _kl in _spectrum_epochs:
            print(f"  epoch {_e}: entropy_corr={_ec:+.3f}  KL={_kl:.3f}")
    _spt = evaluate_disagreement(_best_for_test, test_loader)
    if _spt:
        print(f"Test spectrum: KL(annotators||model)={_spt['kl_q_p']:.3f}, "
              f"entropy corr={_spt['entropy_corr']:.3f}")
del _best_for_test
if "record_metrics" in dir():
    record_metrics(dev_macro_f1=best_f1, test_macro_f1=_test_f1,
                   n_train=len(train_df), n_dev=len(val_df), n_test=len(test_df),
                   dampen_exponent=DAMPEN_EXPONENT)
    if USE_SOFT and "_spt" in dir() and _spt:
        record_metrics(test_spectrum_kl=_spt["kl_q_p"], test_spectrum_entcorr=_spt["entropy_corr"],
                       spectrum_entcorr_by_epoch=";".join(f"{e}:{ec:+.3f}" for e, ec, _ in _spectrum_epochs))


# --- push the freshly trained best checkpoint to the HF Hub so future runs skip training ---
try:
    from huggingface_hub import create_repo
    raise RuntimeError("Hub push disabled for the RoBERTa replication")
    create_repo(MODEL_REPO, exist_ok=True)
    _best = AutoModelForSequenceClassification.from_pretrained(best_model_path)
    _best.push_to_hub(MODEL_REPO)
    tokenizer.push_to_hub(MODEL_REPO)
    print(f"Pushed fine-tuned model to HF Hub: {MODEL_REPO}")
except Exception as e:
    print(f"push_to_hub failed ({e!r}) -- model is still saved locally at ./{best_model_path}")


In [ ]:
# SAVE the trained checkpoint under a clear name (download it from the Output panel)
import os, json, shutil
os.makedirs("/kaggle/working/out", exist_ok=True)
_name = f"roberta_{RUN_LABEL}_{TRAIN_SEED}"
shutil.copy(os.path.join(best_model_path, "model.safetensors"), f"/kaggle/working/out/{_name}.safetensors")
if os.path.exists(RUN_METRICS_PATH):
    shutil.copy(RUN_METRICS_PATH, f"/kaggle/working/out/{_name}_metrics.json")
print("saved", _name, "| test macro-F1 =", round(float(_test_f1), 4))
